In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import pandas as pd
import numpy as np

In [6]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

## Deal with the "additional" stations, which belongs to the network

Two items are excluded, line 2035 (FERN) has been inserted, and 2085, don't know what PC-FWX is (BC-FWX?)

In [7]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl
df_addition = df[
    (df["ISSUE"].str.strip().str.lower() == "additional") 
]

df_addition = df_addition[df_addition["Network ID"].notna()]
df_addition
df_pcds = df_addition[['Network ID' ,'Station ID', 'Unique Names','Unique Location (String)', 'Native ID']].reset_index(drop=True)
df_ted = df_addition[['Station_name','native_id', 'lat', 'lon', 'Elevation']].reset_index(drop=True)


df_pcds['Station_name'] = df_addition['Unique Names'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Unique Names'])

df_pcds['native_id'] = df_addition['Native ID'].str.split('->', n=1, expand=True)[1].reset_index(drop=True).str.strip()
df_pcds = df_pcds.drop(columns=['Native ID'])


In [8]:
df_addition

,ISSUE,Network ID,Network Name,Native ID,Station ID,Unique Names,Unique Location (String),Unique Records,Uniq Obs Freqs,Variables,...,SITE TYPE,OWNER,FIRE CENTRE,FIRE ZONE,LATITUDE,LONGITUDE,ELEVATION,Unnamed: 50,Unnamed: 51,Unnamed: 52
0,Additional,10.0,-> BC AGRIWeather,-> AGOKNGSLFB,NaN,-> Summerland Front Bench,"-> 119.651 W, 49.5794 N, Elev. 426 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Additional,10.0,-> BC AGRIWeather,-> AGAL001,NaN,-> Alkali Lake,"-> 121.2511111 W, 51.79027778 N, Elev. 677 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Additional,10.0,-> BC AGRIWeather,-> AGBCPRDNCK,NaN,-> BCGPA Dawson Creek,"-> 120.143 W, 55.7721 N, Elev. 673 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Additional,10.0,-> BC AGRIWeather,-> AGBCPRBFLT,NaN,-> Bearflats,"-> 121.308 W, 56.2461 N, Elev. 504 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Additional,10.0,-> BC AGRIWeather,-> AGOKNGBVST,NaN,-> Bella Vista,"-> 119.3413 W, 50.2551 N, Elev. 437 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126,Additional,2.0,-> BC-TRAN,-> 34891,NaN,-> Slocan Lake,"-> 117.39829 W. 49.87759 N, Elev. 808 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,Additional,12.0,-> BC-FWx,-> 4332,NaN,-> KASIKS PNG,"-> 129.3252 W, 54.3233 N, Elev. 10 m",NaN,NaN,NaN,...,Project,Auto-Caller,AXIOM/F6,External,BC Wildfire Service,Northwest Fire Centre,Skeena Zone (Kalum),54.3233,-129.3252,10
128,Additional,76.0,-> MVRD-AQ,-> T039,NaN,-> Tsawwassen,"-> 123.082 W, 49.0099 N, Elev 52 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,Additional,5.0,-> BCH-GSO-HMP,-> ALM,NaN,-> Alouette Dam East Embankment,"-> 122.485421 W, 49.285665 N, Elev. 129 m",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
import re

def parse_lat_lon_elev(text):
    # regex to capture: lon value, W/E, lat value, N/S, elevation
    pattern = r'([\d\.]+)\s*([WE]),\s*([\d\.]+)\s*([NS]),\s*Elev\.\s*([\d\.]+)'
    m = re.search(pattern, text)
    
    if not m:
        return None, None, None

    lon_val, lon_dir, lat_val, lat_dir, elev_val = m.groups()

    # Convert numbers
    lon = float(lon_val) * (-1 if lon_dir == "W" else 1)
    lat = float(lat_val) * (-1 if lat_dir == "S" else 1)
    elev = float(elev_val)

    return lat, lon, elev

# Apply to dataframe
df_pcds[['lat', 'lon', 'Elevation']] = df_pcds['Unique Location (String)'].apply(
    lambda x: pd.Series(parse_lat_lon_elev(x))
)
df_pcds = df_pcds.drop(columns=['Unique Location (String)'])

In [10]:
df_pcds.iloc[60:120]

df_ted.head(10)

,Station_name,native_id,lat,lon,Elevation
0,Summerland Front Bench,AGOKNGSLFB,49.5794,-119.651,426
1,Alkali Lake,AGAL001,51.790278,-121.251111,677
2,BCGPA Dawson Creek,AGBCPRDNCK,55.7721,-120.143,673
3,Bearflats,AGBCPRBFLT,56.2461,-121.308,504
4,Bella Vista,AGOKNGBVST,50.2551,-119.3413,437
5,Buick,AGBCPRBUIK,56.7637,-121.085,757
6,Canyon,AGCRESCNYN,49.0853,-116.4475,659
7,Cecil Lake,BCPRCCLK,56.3062,-120.531,718
8,Central Erickson,CRESCEKN,49.0884,-116.4851,648
9,Crescent Island,AGDLTACRES,49.1255,-123.0348,3


In [11]:
# 1. Rename df_pcds columns to match df_ted
df_pcds_renamed = df_pcds.rename(columns={
    'pcds_sation_name': 'Station_name',
    'pcds_native_id': 'native_id',
    'pcds_lat': 'lat',
    'pcds_lon': 'lon',
    'pcds_elev': 'Elevation'
})

df_pcds_renamed = df_pcds_renamed.drop(columns=['Station ID'])

merged = df_pcds_renamed.merge(
    df_ted,
    left_index=True,
    right_index=True,
    how='inner',
    suffixes=('_pcds', '_ted')
)

# 3. Compare key columns
merged['Station_name_match']  = merged['Station_name_pcds'].str.strip()  == merged['Station_name_ted'].str.strip()
merged['lat_match']  = merged['lat_pcds']  == merged['lat_ted']
merged['lon_match']  = merged['lon_pcds']  == merged['lon_ted']
merged['elev_match'] = merged['Elevation_pcds'] == merged['Elevation_ted']
merged['id_match']   = merged['native_id_pcds'].astype(str).str.strip() == merged['native_id_ted'].astype(str).str.strip()

# 4. Whether all match
merged['all_match'] = (
    merged['Station_name_match'] &
    merged['lat_match'] &
    merged['lon_match'] &
    merged['elev_match'] &
    merged['id_match']
)

# 5. Show results
merged[['Station_name_match', 'lat_match', 'lon_match', 'elev_match', 'id_match', 'all_match']]


# df_pcds_renamed.iloc[69]

# merged

,Station_name_match,lat_match,lon_match,elev_match,id_match,all_match
0,True,True,True,True,True,True
1,True,True,True,True,True,True
2,True,True,True,True,True,True
3,True,True,True,True,True,True
4,True,True,True,True,True,True
...,...,...,...,...,...,...
125,True,False,False,False,True,False
126,True,True,True,True,True,True
127,False,False,False,False,False,False
128,True,False,False,False,True,False


In [12]:
mismatched = merged[merged['all_match'] == False]
mismatched[0:50]

# mismatched.to_csv("Sheet1_Addtional_mismatch_44items.csv", index=False)
# mismatched

,Network ID,Station_name_pcds,native_id_pcds,lat_pcds,lon_pcds,Elevation_pcds,Station_name_ted,native_id_ted,lat_ted,lon_ted,Elevation_ted,Station_name_match,lat_match,lon_match,elev_match,id_match,all_match
34,10.0,South ErickSon,AGCRESSEKN,49.080300,-116.493100,650.0,South Erickson,AGCRESSEKN,49.0803,-116.4931,650,False,True,True,True,True,False
48,2.0,Johnson Hill,14093,NaN,NaN,NaN,Port Mann Bridge Johnson Hill,14093,49.20528,-122.80202,93,False,False,False,False,True,False
49,12.0,LITTLE CHOPAKA,NaN,49.025100,-119.690900,461.0,LITTLE CHOPEKA,3810,49.0251,-119.6909,416,False,True,True,False,False,False
50,12.0,TERRACE,NaN,54.438000,-128.571400,199.0,TERRACE,3851,54.438,-128.5714,199,True,True,True,True,False,False
51,12.0,Willis,NaN,49.339400,-120.407100,1455.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,False
52,12.0,BURNS LAKE 850,NaN,54.259100,-125.759900,830.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,False
53,12.0,KOMIE,NaN,59.339400,-122.071800,704.0,KOMIE,4232,59.3394,-122.0718,704,True,True,True,True,False,False
54,12.0,WILDHORSE WX,NaN,51.233300,-122.150000,577.0,NaN,NaN,NaN,NaN,NaN,False,False,False,False,True,False
55,12.0,CHETWYND FB,NaN,55.690700,-121.613700,616.0,CHETWYND,4270,55.6907,-121.6137,616,False,True,True,True,False,False
56,12.0,BLUEBERRY,NaN,53.315900,-119.600700,1088.0,BLUEBERRY,4352,53.3159,-119.6007,1088,True,True,True,True,False,False


In [13]:
len(mismatched)

51

### Going to using the info in the pcds sheet to insert these stations

In [14]:
df_pcds_renamed

save_path = './output_tables/'
df_pcds_renamed.to_csv(save_path + '2.2_130_additional_station_insert_info.csv', index=False)
df_pcds_renamed

,Network ID,Station_name,native_id,lat,lon,Elevation
0,10.0,Summerland Front Bench,AGOKNGSLFB,49.579400,-119.651000,426.0
1,10.0,Alkali Lake,AGAL001,51.790278,-121.251111,677.0
2,10.0,BCGPA Dawson Creek,AGBCPRDNCK,55.772100,-120.143000,673.0
3,10.0,Bearflats,AGBCPRBFLT,56.246100,-121.308000,504.0
4,10.0,Bella Vista,AGOKNGBVST,50.255100,-119.341300,437.0
...,...,...,...,...,...,...
125,2.0,Slocan Lake,34891,NaN,NaN,NaN
126,12.0,KASIKS PNG,4332,54.323300,-129.325200,10.0
127,76.0,Tsawwassen,T039,NaN,NaN,NaN
128,5.0,Alouette Dam East Embankment,ALM,49.285665,-122.485421,129.0


### Insert into db

In [15]:
from sqlalchemy import func

stations_created = []

# for _, row in df_pcds_renamed.iloc[0:2].iterrows():
for _, row in df_pcds_renamed.iterrows():
    
    name = row['Station_name']
    nid  = row['native_id']

    # 1. Create Station
    st = Station(
        native_id=nid,
        publish=True,
        network_id=row['Network ID'])


    session.add(st)
    session.flush()  # 得到 st.id

    h = History(
        station_id=st.id,
        station_name=name,
        lat=row['lat'],
        lon=row['lon'],
        elevation=row['Elevation'],
        province="BC",
        country="CA",
        the_geom=func.ST_SetSRID(func.ST_MakePoint(row['lon'], row['lat']), 4326))

    session.add(h)

    stations_created.append((name, st.id))

session.commit()

print("Created:", stations_created)

Created: [('Summerland Front Bench', 14734), ('Alkali Lake', 14735), ('BCGPA Dawson Creek', 14736), ('Bearflats', 14737), ('Bella Vista', 14738), ('Buick', 14739), ('Canyon', 14740), ('Cecil Lake', 14741), ('Central Erickson', 14742), ('Crescent Island', 14743), ('Farmington', 14744), ('Flatrock', 14745), ('Grandview Flats', 14746), ('Groundbirch', 14747), ('Hullcar Rd East', 14748), ('Hullcar Rd West', 14749), ('Kaleden', 14750), ('Kelly Lake', 14751), ('Keremeos Bipass', 14752), ('Lavington', 14753), ('Merville', 14754), ('Monte Creek', 14755), ('Montney - Bickfords', 14756), ('Naramata Bench', 14757), ('North Crawford Rd', 14758), ('North Erickson', 14759), ('North Grand Forks', 14760), ('North Lister (Riverview)', 14761), ('Osoyoos North', 14762), ('Osoyoos Upper Bench', 14763), ('Permberton Meadows', 14764), ('Prespatou', 14765), ('Rolla', 14766), ('Saanichton', 14767), ('South ErickSon', 14768), ('South Kelowna', 14769), ('Tomslake', 14770), ('Tower Lake', 14771), ('Tsawwassen', 

The insert time is 2026-02-25 10:33:13.203020